## Pitch Rate Control System

In [2]:
import sys
sys.path.append('../')
from modeling.params_f16 import F16Params, Controls
from tools.lin_f16 import get_lin_f16 as linearize
import control as ct
import numpy as np
import matplotlib.pyplot as plt
import tools.control_tools as tools

In [6]:
ALPHA_IDX = 1   # body-axis angle of attack
Q_IDX = 3   # body-axis pitch rate

ELEVATOR_IDX = 1 # elevator control input

RTOD = 57.2958  # radians to degrees
DTOR = 0.0174533  # degrees to radians

In [7]:
params = F16Params()
params.alt_ft = 0.0
params.VT_ftps = 502.0
params.xcg = 0.35
controls = Controls()

In [8]:
lon_sys, _ = linearize(controls, params)

Trim results:
Throttle (0-1): 0.26
Elevator (deg): -0.76
Alpha (deg): 2.12
Aileron (deg): 0.00
Rudder (deg): -0.00
Beta (deg): -0.00


In [9]:
ap = lon_sys.A[[ALPHA_IDX, Q_IDX]][:, [ALPHA_IDX, Q_IDX]]
bp = lon_sys.B[[ALPHA_IDX, Q_IDX]][:,[ELEVATOR_IDX]]
cp = lon_sys.C[[ALPHA_IDX, Q_IDX]][:, [ALPHA_IDX, Q_IDX]] * DTOR
dp = lon_sys.D[[ALPHA_IDX, Q_IDX]] [:,[ELEVATOR_IDX]]
plant = ct.ss(ap, bp, cp, dp)
plant

<LinearIOSystem:sys[6]:['u[0]']->['y[0]', 'y[1]']>

In [ ]:
sysf = ct.ss(-10,10,1,0)
sysa = ct.ss(-20.2,20.2,1,0)
sys1 = ct.series(sysa,plant[ALPHA_IDX,0])
sys2 = ct.series(sys1,sysf)

<LinearICSystem:sys[19]:['u[0]']->['y[0]']>

In [27]:
DE_IDX = 0
ALPHA_IDX = 1
Q_IDX = 2
AF_IDX = 3
EPS_IDX = 4
a = sys2.A
eps_row = np.zeros([1,EPS_IDX+1])
eps_row[0,Q_IDX] = -57.3
a = np.append(a,np.zeros([np.shape(a)[0],1]),axis=1)
a = np.append(a,eps_row,axis=0)
b = sys2.B
b = np.append(b,np.zeros([1,1]),axis=0)
c = np.zeros([3,np.shape(a)[1]])
c[0,AF_IDX] = 57.3
c[1,Q_IDX] = 57.3
c[2,EPS_IDX] = 1
d = np.zeros([np.shape(c)[0],np.shape(b)[1]])
sys3 = ct.ss(a,b,c,d)
sys3

<LinearIOSystem:sys[25]:['u[0]']->['y[0]', 'y[1]', 'y[2]']>

#### Note:
#### $$ \dot{x} = Ax + Bu + Gr$$
#### $$ y = Cx + Fr $$
#### $$ z = Hx $$

In [34]:
A = sys3.A
B = sys3.B
C = sys3.C

g = np.zeros(np.shape(B))
g[EPS_IDX,0] = 1
f = np.zeros([np.shape(C)[0],1])
h = np.zeros([1,np.shape(A)[1]])
h[0,Q_IDX] = 57.3

In [35]:
P_base = abs(h.T @ h)
print('P =\n', P_base)

Q = np.array([[0]])
print('Q =\n', Q)

R = np.array([[1]])  # control effort weight
print('R =\n', R)

P =
 [[   0.      0.      0.      0.      0.  ]
 [   0.      0.      0.      0.      0.  ]
 [   0.      0.   3283.29    0.      0.  ]
 [   0.      0.      0.      0.      0.  ]
 [   0.      0.      0.      0.      0.  ]]
Q =
 [[0]]
R =
 [[1]]


In [41]:
K_0= np.array([[-0.046, -1.072, 3.381]])

K_opt = tools.LQTrackerTime(sys3, g, f, P_base, Q, R, K_0)

J_opt = 1738.3513689865895
K_opt =
 [[ 0.04926277 -0.87932448  3.43661553]]


In [42]:
# K_opt = [[-0.046, -1.072, 3.381]]
A_c = sys3.A - sys3.B @ K_opt @ sys3.C
eigenvalues = np.linalg.eigvals(A_c)
print('Closed-loop eigenvalues:\n', eigenvalues)

Closed-loop eigenvalues:
 [-9.26960905+5.70970661j -9.26960905-5.70970661j -6.34949453+4.12017119j
 -6.34949453-4.12017119j -1.05742788+0.j        ]
